# Data Preparation for Model Training

## Step 1: Data Loading & Exploration

In [26]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load the data
df = pd.read_csv('data.csv')



## Step 3: Feature Analysis

In [27]:
# Identify categorical and numerical columns
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Remove target variable from numerical columns if it exists
if 'Score' in numerical_cols:
    numerical_cols.remove('Score')

print("Categorical Columns:", categorical_cols)
print("Numerical Columns:", numerical_cols)
print("\nUnique values in categorical columns:")
for col in categorical_cols:
    print(f"{col}: {df[col].unique()}")

Categorical Columns: ['Platform', 'Content_Type', 'Category', 'Day_of_Week', 'Sentiment']
Numerical Columns: ['Follower_Count', 'Hour_of_Day', 'Hashtag_Count', 'Content_Length']

Unique values in categorical columns:
Platform: ['Instagram' 'LinkedIn' 'Facebook' 'Twitter' 'YouTube' 'TikTok']
Content_Type: ['Carousel' 'Document' 'Video' 'Photo' 'Poll' 'Thread' 'Live' 'Reel'
 'Post' 'Story' 'Short' 'Tweet' 'Retweet' 'Article' 'Duet'
 'Community Post' 'Stitch']
Category: ['Business' 'Health' 'Food' 'Sports' 'Fitness' 'Lifestyle' 'Education'
 'Travel' 'Entertainment' 'Technology' 'Gaming' 'Fashion']
Day_of_Week: ['Monday' 'Tuesday' 'Wednesday' 'Thursday' 'Friday' 'Saturday' 'Sunday']
Sentiment: ['Positive' 'Negative' 'Neutral']


## Step 4: Configure Categorical Columns for One-Hot Encoding

In [28]:
# Categorical columns to one-hot encode (as requested)
categorical_cols = ['Platform', 'Content_Type', 'Category', 'Day_of_Week', 'Sentiment']

# Keep only columns that actually exist in the dataset
categorical_cols = [col for col in categorical_cols if col in df.columns]

print("Categorical columns selected for one-hot encoding:", categorical_cols)

Categorical columns selected for one-hot encoding: ['Platform', 'Content_Type', 'Category', 'Day_of_Week', 'Sentiment']


## Step 5: Features & Target Separation

In [29]:
# Separate features and target (before encoding)
X = df.drop('Score', axis=1)
y = df['Score']

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("\nFeature columns:", X.columns.tolist())
print("\nTarget variable statistics:")
print(y.describe())

Features shape: (4913, 11)
Target shape: (4913,)

Feature columns: ['Platform', 'Content_Type', 'Category', 'Follower_Count', 'Hour_of_Day', 'Day_of_Week', 'Hashtag_Count', 'Content_Length', 'Sentiment', 'Has_Media', 'Is_Verified']

Target variable statistics:
count    4913.000000
mean        0.532982
std         0.191031
min         0.000000
25%         0.406973
50%         0.501575
75%         0.618891
max         1.000000
Name: Score, dtype: float64


## Step 6: Train-Test Split (Before Encoding)

In [30]:
# Split data into training and testing sets (80-20 split)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training set size:", X_train.shape)
print("Testing set size:", X_test.shape)
print("\nTraining target statistics:")
print(y_train.describe())
print("\nTesting target statistics:")
print(y_test.describe())

Training set size: (3930, 11)
Testing set size: (983, 11)

Training target statistics:
count    3930.000000
mean        0.534886
std         0.191222
min         0.000000
25%         0.408782
50%         0.503336
75%         0.620543
max         1.000000
Name: Score, dtype: float64

Testing target statistics:
count    983.000000
mean       0.525369
std        0.190173
min        0.073552
25%        0.395615
50%        0.498589
75%        0.612582
max        0.989540
Name: Score, dtype: float64


## Step 7: One-Hot Encode and Align Train/Test Features

In [31]:
# Apply one-hot encoding AFTER train/test split
X_train = pd.get_dummies(X_train, columns=categorical_cols, drop_first=True)
X_test = pd.get_dummies(X_test, columns=categorical_cols, drop_first=True)

# Align test columns to training columns and fill missing columns with 0
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

print("Encoded X_train shape:", X_train.shape)
print("Encoded X_test shape:", X_test.shape)
print("\nX_train and X_test are aligned and ready for XGBoost.")

Encoded X_train shape: (3930, 46)
Encoded X_test shape: (983, 46)

X_train and X_test are aligned and ready for XGBoost.


## Step 8: Data Preparation Summary

In [32]:
print("DATA PREPARATION SUMMARY")
print(f"Total records: {len(df)}")
print(f"Original features: {X.shape[1]}")
print(f"Encoded training features: {X_train.shape[1]}")
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"Training set: {X_train.shape[0]} samples ({X_train.shape[0] / len(df) * 100:.1f}%)")
print(f"Testing set: {X_test.shape[0]} samples ({X_test.shape[0] / len(df) * 100:.1f}%)")
print(f"\nFirst 15 encoded features: {', '.join(X_train.columns.tolist()[:15])}")
print("\nX_train and X_test are now ready for XGBoost.")

DATA PREPARATION SUMMARY
Total records: 4913
Original features: 11
Encoded training features: 46
Missing values: 0
Training set: 3930 samples (80.0%)
Testing set: 983 samples (20.0%)

First 15 encoded features: Follower_Count, Hour_of_Day, Hashtag_Count, Content_Length, Has_Media, Is_Verified, Platform_Instagram, Platform_LinkedIn, Platform_TikTok, Platform_Twitter, Platform_YouTube, Content_Type_Carousel, Content_Type_Community Post, Content_Type_Document, Content_Type_Duet

X_train and X_test are now ready for XGBoost.


## Step 9: Train and Evaluate XGBoost

In [33]:
# Train an XGBoost regressor on the processed features
xgb_model = XGBRegressor(
    objective='reg:squarederror',
    n_estimators=400,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)

# Predictions for train and test sets
y_train_pred = xgb_model.predict(X_train)
y_test_pred = xgb_model.predict(X_test)

# Regression metrics
train_mae = mean_absolute_error(y_train, y_train_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
train_r2 = r2_score(y_train, y_train_pred)

test_mae = mean_absolute_error(y_test, y_test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
test_r2 = r2_score(y_test, y_test_pred)

print('XGBOOST REGRESSION EVALUATION')
print(f'Train MAE:  {train_mae:.4f}')
print(f'Train RMSE: {train_rmse:.4f}')
print(f'Train R2:   {train_r2:.4f}')
print('')
print(f'Test MAE:   {test_mae:.4f}')
print(f'Test RMSE:  {test_rmse:.4f}')
print(f'Test R2:    {test_r2:.4f}')

XGBOOST REGRESSION EVALUATION
Train MAE:  0.0358
Train RMSE: 0.0464
Train R2:   0.9412

Test MAE:   0.0759
Test RMSE:  0.0966
Test R2:    0.7416
